# Grammar Scoring Engine for Spoken Audio (v4)

**Task:** predict a continuous grammar score (0-5) from a 45-60 s speech clip. **Metric:** RMSE (lower = better).

## Version history
| version | public RMSE | notes |
|---|---|---|
| v2 | 0.4062 | 3 encoders, stacking |
| v3 | 0.3966 | + WavLM-large, CTC features, 5 seeds |
| **v4** | ? | + HuBERT-large, wav2vec2-large-lv60, Whisper-medium encoder; calibration chosen by nested CV; dead models removed |

## What is new in v4 (driven by the v3 diagnostics)
* v3 showed large encoders dominate (WavLM-large: best single model, biggest stack weight) and zero-label clips are already solved (0.4% of squared error). So v4 adds **more large encoders** (Section 5.6b, each toggled in `EXTRA_ENCODERS`, each cached separately; a failing model is skipped, not fatal).
* **Mean + std pooling** was already used for every layer of every encoder (see `pool_layers`), so nothing to add there.
* **Post-hoc calibration** (none / linear / isotonic) picked per seed by nested CV, aimed at the over-prediction of low scores seen in v3.
* **Removed** kernel ridge and the MLP (near-zero stack weight, worst RMSE); their compute goes to the new encoders.

**Evaluation.** Stratified 5-fold CV, out-of-fold (OOF) RMSE is the honest estimate. Optimise CV, not the 216-row public leaderboard.

## 1. Setup

In [ ]:
# !pip install -q librosa soundfile transformers torch scikit-learn lightgbm matplotlib seaborn scipy tqdm
import os, re, gc, json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from tqdm.auto import tqdm
from scipy.stats import pearsonr, spearmanr
import librosa, torch

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

In [ ]:
!pip install -q sentence-transformers

## 2. Configuration

Train and test ids overlap in file *name* (e.g. `audio_0.wav` exists in both) but are different recordings, so audio is read from **separate folders** and rows are never merged by file name.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# CSVs you uploaded directly to Colab (they land in /content/)
CSV_DIR = "/content"

# Audio folders on Google Drive: change these to your real folder names
TRAIN_AUDIO = "/content/drive/MyDrive/train"
TEST_AUDIO  = "/content/drive/MyDrive/test"

# Save the cache on Drive so you don't lose transcripts/features if Colab disconnects
CACHE_DIR = "/content/drive/MyDrive/grammar/cache"; os.makedirs(CACHE_DIR, exist_ok=True)

SR = 16_000
ASR_SIZE   = "small"      # "small" reuses your existing v2 caches. "medium" = better transcripts but ~2-3x slower
                          # (if you change it, use a NEW CACHE_DIR so old transcripts are not reused)
ASR_MODEL  = f"openai/whisper-{ASR_SIZE}.en" if DEVICE == "cuda" else "openai/whisper-base.en"
# Extra encoders (all layers, mean+std pooled). Delete a line to skip it. Each is cached on Drive separately.
# "wavlm_large" reuses the cache you already built in v3. Each new one costs ~8-20 min on a T4 the first time.
EXTRA_ENCODERS = {
    "wavlm_large":  ("microsoft/wavlm-large",            "hf"),
    "hubert_large": ("facebook/hubert-large-ll60k",      "hf"),
    "w2v_large":    ("facebook/wav2vec2-large-lv60",     "hf"),
    "whisper_med":  ("openai/whisper-medium.en",         "whisper"),
}
USE_CTC_FEATS   = True    # verbatim-transcript / confidence features
SEEDS = [42, 43, 44, 45, 46]   # multi-seed ensemble
W2V_MODEL  = "facebook/wav2vec2-base-960h"
W2V_LAYER  = 8
LM_MODEL   = "gpt2"

## 3. Data loading & exploratory analysis

In [ ]:
train = pd.read_csv(f"{CSV_DIR}/train.csv")
test  = pd.read_csv(f"{CSV_DIR}/test.csv")
print(train.shape, test.shape)
train["path"] = TRAIN_AUDIO + "/" + train["filename"]
test["path"]  = TEST_AUDIO  + "/" + test["filename"]
missing = [p for p in list(train.path) + list(test.path) if not os.path.exists(p)]
print("missing audio files:", len(missing)); assert not missing, missing[:5]
train.describe()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(x="label", data=train, ax=ax[0], color="steelblue")
ax[0].set_title("Label distribution (train)")
sns.histplot(train["label"], bins=20, kde=True, ax=ax[1]); ax[1].set_title("Label density")
plt.tight_layout(); plt.show()
print(train["label"].value_counts().sort_index())

Labels lie on a 0.5 grid. The rubric defines 1–5, but 37 training clips are labelled **0.0**; the next section checks what those clips look like.

## 4. Audio preprocessing
Mono, 16 kHz, silence-trimmed, peak-normalised; prosody features from an energy-based voice-activity split.

In [ ]:
def load_audio(path, sr=SR):
    # Load as mono 16 kHz float32, trim leading/trailing silence, peak-normalise.
    y, _ = librosa.load(path, sr=sr, mono=True)
    y, _ = librosa.effects.trim(y, top_db=35)
    peak = np.max(np.abs(y)) if len(y) else 0
    return (y / peak).astype(np.float32) if peak > 0 else y.astype(np.float32)

def prosody_features(y, sr=SR):
    # Pause / rate / energy features from a simple energy-based voice-activity split.
    dur = len(y) / sr if len(y) else 1e-6
    iv = librosa.effects.split(y, top_db=30, frame_length=1024, hop_length=256) if len(y) else np.zeros((0, 2), int)
    speech = sum(e - s for s, e in iv) / sr
    gaps = np.array([(iv[i + 1][0] - iv[i][1]) / sr for i in range(len(iv) - 1)]) if len(iv) > 1 else np.array([0.0])
    rms = librosa.feature.rms(y=y)[0] if len(y) else np.array([0.0])
    mf = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13) if len(y) else np.zeros((13, 1))
    f = dict(duration=dur, speech_ratio=speech / dur, n_segments=len(iv), seg_per_sec=len(iv) / dur,
             pause_mean=gaps.mean(), pause_max=gaps.max(), n_long_pauses=int((gaps > 0.5).sum()),
             long_pause_rate=(gaps > 0.5).sum() / dur, rms_mean=rms.mean(), rms_std=rms.std())
    f.update({f"mfcc{i}_mean": mf[i].mean() for i in range(13)})
    f.update({f"mfcc{i}_std": mf[i].std() for i in range(13)})
    return f

## 5. Feature extraction
### 5.1 Transcription with Whisper (cached on Drive)

In [ ]:
from transformers import pipeline
asr = pipeline("automatic-speech-recognition", model=ASR_MODEL, device=0 if DEVICE == "cuda" else -1,
               chunk_length_s=30, torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32)

def transcribe(df, name):
    cache = f"{CACHE_DIR}/{name}_transcripts.json"
    if os.path.exists(cache):
        return json.load(open(cache))
    out = []
    for p in tqdm(df.path, desc=f"ASR {name}"):
        y = load_audio(p)
        out.append(asr({"raw": y, "sampling_rate": SR})["text"].strip() if len(y) > SR else "")
    json.dump(out, open(cache, "w")); return out

train["text"] = transcribe(train, "train")
test["text"]  = transcribe(test, "test")
train[["filename", "label", "text"]].head()

In [ ]:
# Sanity check: are the 0-labelled clips really "no usable speech"?
train["n_words"] = train.text.str.split().str.len()
print(train.groupby(train.label == 0)["n_words"].describe())

**Finding:** zero-labelled clips are *not* short or silent – their word counts are similar to the rest (some are extreme, up to ~700 words, which suggests Whisper looping). A word-count rule cannot find them, so v2 adds transcript-loop detectors and lets a classifier try on embeddings (Section 6.3).

### 5.2 Transcript-based grammar / fluency features

In [ ]:
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
lm_tok = GPT2TokenizerFast.from_pretrained(LM_MODEL)
lm = GPT2LMHeadModel.from_pretrained(LM_MODEL).to(DEVICE).eval()

@torch.no_grad()
def perplexity_stats(text, max_len=512):
    # GPT-2 token-level NLL: mean, std and 90th percentile (a few very surprising tokens = likely errors).
    ids = lm_tok(text, return_tensors="pt", truncation=True, max_length=max_len).input_ids.to(DEVICE)
    if ids.shape[1] < 3: return dict(ppl=np.nan, nll_std=np.nan, nll_p90=np.nan)
    logits = lm(ids).logits[0, :-1]
    nll = torch.nn.functional.cross_entropy(logits, ids[0, 1:], reduction="none").cpu().numpy()
    return dict(ppl=float(np.exp(nll.mean())), nll_std=float(nll.std()), nll_p90=float(np.percentile(nll, 90)))

FILLERS = re.compile(r"\b(um+|uh+|er+|ah+|hmm+|like|you know|i mean|kind of|sort of)\b", re.I)
AUX = re.compile(r"\b(is|are|was|were|be|been|being|have|has|had|do|does|did|will|would|can|could|should|may|might)\b", re.I)
SUBORD = re.compile(r"\b(because|although|though|while|whereas|if|unless|which|that|who|whom|whose|when|since|so that)\b", re.I)

def text_features(text):
    words = re.findall(r"[A-Za-z']+", text.lower())
    n = len(words)
    sents = [s for s in re.split(r"[.!?]+", text) if s.strip()]
    slen = np.array([len(s.split()) for s in sents]) if sents else np.array([0])
    reps = sum(words[i] == words[i + 1] for i in range(n - 1)) if n > 1 else 0            # "the the"
    bigrams = list(zip(words, words[1:]))
    f = dict(n_words=n, n_unique=len(set(words)), ttr=len(set(words)) / max(n, 1),
             n_sents=len(sents), sent_len_mean=slen.mean(), sent_len_std=slen.std(), sent_len_max=slen.max(),
             avg_word_len=np.mean([len(w) for w in words]) if n else 0,
             long_word_ratio=np.mean([len(w) > 6 for w in words]) if n else 0,
             filler_ratio=len(FILLERS.findall(text)) / max(n, 1),
             repeat_ratio=reps / max(n, 1),
             bigram_repeat=1 - len(set(bigrams)) / max(len(bigrams), 1),
             aux_ratio=len(AUX.findall(text)) / max(n, 1),
             subord_ratio=len(SUBORD.findall(text)) / max(n, 1),
             comma_ratio=text.count(",") / max(n, 1))
    f.update(perplexity_stats(text) if n else dict(ppl=np.nan, nll_std=np.nan, nll_p90=np.nan))
    return f

def build_text_feats(df, name):
    cache = f"{CACHE_DIR}/{name}_text_feats.parquet"
    if os.path.exists(cache): return pd.read_parquet(cache)
    out = pd.DataFrame([text_features(t) for t in tqdm(df.text, desc=f"text {name}")]); out.to_parquet(cache); return out

Xt_train, Xt_test = build_text_feats(train, "train"), build_text_feats(test, "test")
Xt_train.describe().T.head(12)

### 5.3 v1 acoustic features: prosody + wav2vec 2.0 (layer 8)

In [ ]:
from transformers import Wav2Vec2Processor, Wav2Vec2Model
w2v_proc = Wav2Vec2Processor.from_pretrained(W2V_MODEL)
w2v = Wav2Vec2Model.from_pretrained(W2V_MODEL).to(DEVICE).eval()

@torch.no_grad()
def w2v_embed(y):
    # mean + std pooling of one hidden layer over time -> 1536-d vector
    if len(y) < SR // 2: return np.zeros(2 * 768, np.float32)
    x = w2v_proc(y, sampling_rate=SR, return_tensors="pt").input_values.to(DEVICE)
    h = w2v(x, output_hidden_states=True).hidden_states[W2V_LAYER][0]
    return torch.cat([h.mean(0), h.std(0)]).cpu().numpy()

def build_audio_feats(df, name):
    cache = f"{CACHE_DIR}/{name}_audio_feats.npz"
    if os.path.exists(cache):
        z = np.load(cache, allow_pickle=True); return pd.DataFrame(z["pros"], columns=z["cols"]), z["emb"]
    pros, emb = [], []
    for p in tqdm(df.path, desc=f"audio {name}"):
        y = load_audio(p); pros.append(prosody_features(y)); emb.append(w2v_embed(y))
    pros = pd.DataFrame(pros); emb = np.vstack(emb)
    np.savez(cache, pros=pros.values, cols=np.array(pros.columns), emb=emb); return pros, emb

Xp_train, E_train = build_audio_feats(train, "train")
Xp_test,  E_test  = build_audio_feats(test, "test")
print(Xp_train.shape, E_train.shape, Xp_test.shape, E_test.shape)

del asr, lm, w2v; gc.collect(); torch.cuda.empty_cache() if DEVICE == "cuda" else None

### 5.4 v1 feature-vs-label analysis

In [ ]:
H_train = pd.concat([Xt_train, Xp_train], axis=1); H_test = pd.concat([Xt_test, Xp_test], axis=1)
H_train = H_train.replace([np.inf, -np.inf], np.nan); H_test = H_test.replace([np.inf, -np.inf], np.nan)
med = H_train.median()
H_train, H_test = H_train.fillna(med), H_test.fillna(med)          # median imputation (train medians only)

corr = H_train.corrwith(train.label, method="spearman").dropna()
top = corr.reindex(corr.abs().sort_values(ascending=False).index)[:15]
plt.figure(figsize=(7, 5)); top[::-1].plot.barh(color=["tomato" if v < 0 else "seagreen" for v in top[::-1]])
plt.title("Top handcrafted features: Spearman correlation with grammar score"); plt.tight_layout(); plt.show()

### 5.5 NEW – multi-layer embeddings from three speech encoders
Stored for *every* layer so the best layers can be selected by cross-validation. Cached on Drive; roughly 10–25 min the first time on a T4 depending on Drive read speed.

In [ ]:
# Three complementary speech encoders, all run in ONE pass over each audio file.
# For every hidden layer we keep mean+std pooling over time -> array (n_files, n_layers, 2*hidden).
# Keeping all layers lets Section 6 pick the best layers by cross-validation instead of guessing.
from transformers import (Wav2Vec2Model, AutoFeatureExtractor, WavLMModel,
                          WhisperFeatureExtractor, WhisperModel)

WDT = torch.float16 if DEVICE == "cuda" else torch.float32
w2v_m    = Wav2Vec2Model.from_pretrained(W2V_MODEL).to(DEVICE).eval()
wavlm_fe = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-base-plus")
wavlm_m  = WavLMModel.from_pretrained("microsoft/wavlm-base-plus").to(DEVICE).eval()
whis_fe  = WhisperFeatureExtractor.from_pretrained(ASR_MODEL)
whis_enc = WhisperModel.from_pretrained(ASR_MODEL, torch_dtype=WDT).encoder.to(DEVICE).eval()

def pool_layers(hidden_states, n_frames=None):
    rows = []
    for h in hidden_states:
        h = h[0].float()
        if n_frames is not None: h = h[:n_frames]
        sd = h.std(0) if h.shape[0] > 1 else torch.zeros_like(h[0])
        rows.append(torch.cat([h.mean(0), sd]))
    return torch.stack(rows).cpu().numpy()

@torch.no_grad()
def emb_hf(model, fe, y):
    L, D = model.config.num_hidden_layers + 1, 2 * model.config.hidden_size
    if len(y) < SR // 2: return np.zeros((L, D), np.float32)
    x = fe(y, sampling_rate=SR, return_tensors="pt").input_values.to(DEVICE)
    return pool_layers(model(x, output_hidden_states=True).hidden_states)

@torch.no_grad()
def emb_whisper(y):
    # Whisper's encoder only accepts 30 s windows -> process 30 s chunks and average them.
    L, D = whis_enc.config.encoder_layers + 1, 2 * whis_enc.config.d_model
    parts = []
    for s in range(0, len(y), 30 * SR):
        seg = y[s:s + 30 * SR]
        if len(seg) < SR: continue
        feats = whis_fe(seg, sampling_rate=SR, return_tensors="pt").input_features.to(DEVICE, WDT)
        n_frames = int(np.ceil(len(seg) / SR * 50))            # encoder output = 50 frames / second
        parts.append(pool_layers(whis_enc(feats, output_hidden_states=True).hidden_states, n_frames))
    return np.mean(parts, 0) if parts else np.zeros((L, D), np.float32)

def extract_all(df, name):
    cache = f"{CACHE_DIR}/{name}_layers_v2.npz"
    if os.path.exists(cache):
        z = np.load(cache); return {k: z[k] for k in z.files}
    res = {"w2v": [], "wavlm": [], "whisper": []}
    for p in tqdm(df.path, desc=f"embeddings {name}"):
        y_ = load_audio(p)
        res["w2v"].append(emb_hf(w2v_m, w2v_proc, y_))
        res["wavlm"].append(emb_hf(wavlm_m, wavlm_fe, y_))
        res["whisper"].append(emb_whisper(y_))
    res = {k: np.stack(v) for k, v in res.items()}
    np.savez(cache, **res); return res

emb_train = extract_all(train, "train")
emb_test  = extract_all(test,  "test")
print({k: v.shape for k, v in emb_train.items()})

del w2v_m, wavlm_m, whis_enc; gc.collect()
if DEVICE == "cuda": torch.cuda.empty_cache()

### 5.6 NEW – grammatical-acceptability (CoLA) and transcript embeddings

In [ ]:
# (a) Sentence-level grammatical acceptability (CoLA-trained RoBERTa): a direct grammar signal.
# (b) Sentence-transformer embedding of the whole transcript (syntax + vocabulary level).
from transformers import pipeline
from sentence_transformers import SentenceTransformer

cola = pipeline("text-classification", model="textattack/roberta-base-CoLA", device=0 if DEVICE == "cuda" else -1)

def chunks(text, n=18):
    # Whisper output often lacks punctuation, so long runs are cut into ~18-word pieces.
    out = []
    for s in [s.strip() for s in re.split(r"[.!?]+", text) if len(s.split()) >= 3]:
        w = s.split()
        out += [" ".join(w[i:i + n]) for i in range(0, len(w), n)] if len(w) > n * 1.5 else [s]
    return out or [text if text.strip() else "empty"]

def cola_feats(text):
    res = cola(chunks(text), batch_size=32, truncation=True, max_length=64)
    p = np.array([r["score"] if r["label"] == "LABEL_1" else 1 - r["score"] for r in res])
    return dict(cola_mean=p.mean(), cola_min=p.min(), cola_std=p.std(),
                cola_p25=np.percentile(p, 25), cola_bad=(p < 0.5).mean(), n_chunks=len(p))

def build_cola(df, name):
    cache = f"{CACHE_DIR}/{name}_cola.parquet"
    if os.path.exists(cache): return pd.read_parquet(cache)
    out = pd.DataFrame([cola_feats(t) for t in tqdm(df.text, desc=f"CoLA {name}")]); out.to_parquet(cache); return out

Xc_train, Xc_test = build_cola(train, "train"), build_cola(test, "test")

st = SentenceTransformer("all-mpnet-base-v2", device=DEVICE)
def build_txt_emb(df, name):
    cache = f"{CACHE_DIR}/{name}_txtemb.npy"
    if os.path.exists(cache): return np.load(cache)
    e = st.encode(df.text.tolist(), batch_size=32, show_progress_bar=True, normalize_embeddings=True)
    np.save(cache, e); return e

T_train, T_test = build_txt_emb(train, "train"), build_txt_emb(test, "test")
del cola, st; gc.collect()
if DEVICE == "cuda": torch.cuda.empty_cache()
print(Xc_train.shape, T_train.shape)
Xc_train.describe().T

### 5.6b NEW - Extra large encoders (all layers, mean+std pooling)

In [ ]:
from transformers import AutoModel
ENCODERS = ["w2v", "wavlm", "whisper"]

def get_fe(mid):
    try: return AutoFeatureExtractor.from_pretrained(mid)
    except Exception:
        from transformers import Wav2Vec2FeatureExtractor
        return Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=SR, padding_value=0.0, do_normalize=True, return_attention_mask=False)

@torch.no_grad()
def emb_whisper_generic(enc, fe, y_):
    L, D = enc.config.encoder_layers + 1, 2 * enc.config.d_model
    parts = []
    for s0 in range(0, len(y_), 30 * SR):
        seg = y_[s0:s0 + 30 * SR]
        if len(seg) < SR: continue
        feats = fe(seg, sampling_rate=SR, return_tensors="pt").input_features.to(DEVICE, WDT)
        n_frames = int(np.ceil(len(seg) / SR * 50))
        parts.append(pool_layers(enc(feats, output_hidden_states=True).hidden_states, n_frames))
    return np.mean(parts, 0) if parts else np.zeros((L, D), np.float32)

for key, (mid, kind) in EXTRA_ENCODERS.items():
    tag = {"wavlm_large": "wavlmL"}.get(key, key)          # keeps the v3 cache file name for WavLM-large
    paths = {nm: f"{CACHE_DIR}/{nm}_layers_{tag}.npy" for nm in ("train", "test")}
    try:
        if not all(os.path.exists(p) for p in paths.values()):
            if kind == "hf":
                fe = get_fe(mid); mdl = AutoModel.from_pretrained(mid).to(DEVICE).eval()
                fn = lambda y_: emb_hf(mdl, fe, y_)
            else:
                fe = WhisperFeatureExtractor.from_pretrained(mid)
                mdl = WhisperModel.from_pretrained(mid, torch_dtype=WDT).encoder.to(DEVICE).eval()
                fn = lambda y_: emb_whisper_generic(mdl, fe, y_)
            for nm, df in (("train", train), ("test", test)):
                arr = np.stack([fn(load_audio(p)) for p in tqdm(df.path, desc=f"{key} {nm}")])
                np.save(paths[nm], arr)
            del mdl; gc.collect()
            if DEVICE == "cuda": torch.cuda.empty_cache()
        emb_train[key], emb_test[key] = np.load(paths["train"]), np.load(paths["test"])
        ENCODERS.append(key)
    except Exception as e:
        print(f"[skip {key}] {type(e).__name__}: {e}")
        gc.collect()
        if DEVICE == "cuda": torch.cuda.empty_cache()
print(ENCODERS); print({k: v.shape for k, v in emb_train.items()})

### 5.7 NEW - CTC (verbatim) features: confidence, entropy, Whisper-vs-CTC disagreement
Whisper 'repairs' errors and hallucinates on bad audio. The wav2vec2 CTC decoder does neither, so the *disagreement* between the two and the CTC confidence are strong signals for unclear / broken / off-task clips (including the zero-label ones).

In [ ]:
!pip install -q jiwer
import jiwer
from transformers import Wav2Vec2ForCTC

CTC_COLS = ["ctc_conf", "ctc_ent", "ctc_blank", "ctc_wer", "ctc_nwords", "ctc_ppl", "ctc_nll_p90"]
if USE_CTC_FEATS:
    ctc = Wav2Vec2ForCTC.from_pretrained(W2V_MODEL).to(DEVICE).eval()
    lm2 = GPT2LMHeadModel.from_pretrained(LM_MODEL).to(DEVICE).eval()
    norm = lambda t: re.sub(r"[^A-Z' ]", "", t.upper()).strip()

    @torch.no_grad()
    def ctc_feats(path, whisper_text):
        y_ = load_audio(path)
        if len(y_) < SR // 2:
            return dict(ctc_conf=0, ctc_ent=0, ctc_blank=1, ctc_wer=1, ctc_nwords=0, ctc_ppl=np.nan, ctc_nll_p90=np.nan)
        x = w2v_proc(y_, sampling_rate=SR, return_tensors="pt").input_values.to(DEVICE)
        lg = ctc(x).logits[0].float()
        p = lg.softmax(-1)
        txt = w2v_proc.batch_decode(lg.argmax(-1)[None])[0].lower()
        ref = norm(whisper_text) or "EMPTY"
        f = dict(ctc_conf=p.max(-1).values.mean().item(),
                 ctc_ent=-(p * (p + 1e-9).log()).sum(-1).mean().item(),
                 ctc_blank=(lg.argmax(-1) == 0).float().mean().item(),
                 ctc_wer=min(jiwer.wer(ref, txt.upper() or "EMPTY"), 3.0),
                 ctc_nwords=len(txt.split()))
        # GPT-2 perplexity of the VERBATIM transcript (errors are not repaired)
        ids = lm_tok(txt, return_tensors="pt", truncation=True, max_length=512).input_ids.to(DEVICE)
        if ids.shape[1] >= 3:
            nll = torch.nn.functional.cross_entropy(lm2(ids).logits[0, :-1], ids[0, 1:], reduction="none").cpu().numpy()
            f.update(ctc_ppl=float(np.exp(nll.mean())), ctc_nll_p90=float(np.percentile(nll, 90)))
        else:
            f.update(ctc_ppl=np.nan, ctc_nll_p90=np.nan)
        return f

    def build_ctc(df, name):
        cache = f"{CACHE_DIR}/{name}_ctc.parquet"
        if os.path.exists(cache): return pd.read_parquet(cache)
        out = pd.DataFrame([ctc_feats(p, t) for p, t in tqdm(zip(df.path, df.text), total=len(df), desc=f"CTC {name}")])
        out.to_parquet(cache); return out

    Xctc_train, Xctc_test = build_ctc(train, "train"), build_ctc(test, "test")
    del ctc, lm2; gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()
    print(Xctc_train.describe().T.round(3))
    print("Spearman with label:\n", Xctc_train.corrwith(train.label, method="spearman").round(3))
else:
    Xctc_train = pd.DataFrame(index=train.index); Xctc_test = pd.DataFrame(index=test.index)

### 5.8 Assemble the handcrafted feature matrix

In [ ]:
import zlib
# Two cheap "broken transcript" detectors: Whisper loops/hallucinates on some clips (see the 699-word max above),
# which shows up as highly compressible text and an implausible words-per-second rate.
def extra_text_feats(text, dur):
    b = text.encode()
    return dict(compress_ratio=len(b) / max(len(zlib.compress(b)), 1), wps=len(text.split()) / max(dur, 1e-6))

Xe_train = pd.DataFrame([extra_text_feats(t, d) for t, d in zip(train.text, Xp_train["duration"])])
Xe_test  = pd.DataFrame([extra_text_feats(t, d) for t, d in zip(test.text,  Xp_test["duration"])])

H_train = pd.concat([Xt_train, Xp_train, Xc_train, Xe_train, Xctc_train], axis=1).replace([np.inf, -np.inf], np.nan)
H_test  = pd.concat([Xt_test,  Xp_test,  Xc_test,  Xe_test,  Xctc_test],  axis=1).replace([np.inf, -np.inf], np.nan)
med = H_train.median()
H_train, H_test = H_train.fillna(med), H_test.fillna(med)          # train medians only
print("handcrafted feature matrix:", H_train.shape, H_test.shape)

corr = H_train.corrwith(train.label, method="spearman").dropna()
top = corr.reindex(corr.abs().sort_values(ascending=False).index)[:15]
plt.figure(figsize=(7, 5)); top[::-1].plot.barh(color=["tomato" if v < 0 else "seagreen" for v in top[::-1]])
plt.title("Top handcrafted features (incl. CoLA): Spearman vs grammar score"); plt.tight_layout(); plt.show()

## 6. Modelling
### 6.1 Cross-validation setup and layer selection

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeCV, LinearRegression, LogisticRegression
from sklearn.svm import SVR
from sklearn.kernel_ridge import KernelRidge
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.metrics import mean_squared_error, roc_auc_score
try:
    from lightgbm import LGBMRegressor; HAS_LGBM = True
except Exception:
    HAS_LGBM = False
print("LightGBM available:", HAS_LGBM)

y = train.label.values
rmse = lambda a, b: float(np.sqrt(mean_squared_error(a, b)))
strata = np.round(y * 2).astype(int)
def make_folds(seed): return list(StratifiedKFold(5, shuffle=True, random_state=seed).split(np.zeros(len(y)), strata))
FOLDS = make_folds(SEED)

def ridge_model(): return make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(1, 5, 15)))

def cv_oof(make_model, X):
    p = np.zeros(len(y))
    for tr, va in FOLDS:
        p[va] = np.clip(make_model().fit(X[tr], y[tr]).predict(X[va]), 0, 5)
    return p

In [ ]:
# Which layer of which encoder carries the most grammar/proficiency information?
layer_rmse = {s: [rmse(y, cv_oof(ridge_model, emb_train[s][:, l])) for l in range(emb_train[s].shape[1])] for s in ENCODERS}

plt.figure(figsize=(9, 4))
for s, v in layer_rmse.items(): plt.plot(v, marker="o", ms=3, label=s)
plt.xlabel("hidden layer"); plt.ylabel("OOF RMSE (ridge)"); plt.title("Layer scan: lower is better"); plt.legend(); plt.tight_layout(); plt.show()

TOPK = 3
best_layers = {s: sorted(np.argsort(v)[:TOPK].tolist()) for s, v in layer_rmse.items()}
print("best layers:", best_layers)
print({s: round(min(v), 4) for s, v in layer_rmse.items()}, "<- best single-layer OOF RMSE per encoder")

### 6.2 Feature views and base learners
Ridge suits high-dimensional embeddings; SVR / kernel ridge / MLP add non-linear views; boosting handles handcrafted features; the ordinal model predicts the expected value of a classifier over the 0.5-grid labels.

In [ ]:
def layer_view(arr, s): return arr[:, best_layers[s]].mean(1)

views_train = {s: layer_view(emb_train[s], s) for s in ENCODERS}
views_test  = {s: layer_view(emb_test[s],  s) for s in ENCODERS}
views_train.update(emb=E_train, txt=T_train, hand=H_train.values)          # emb = v1 wav2vec2 layer 8
views_test.update(emb=E_test, txt=T_test, hand=H_test.values)
CONCAT_KEYS = ENCODERS + ["txt"]
views_train["concat"] = np.hstack([views_train[k] for k in CONCAT_KEYS])
views_test["concat"]  = np.hstack([views_test[k]  for k in CONCAT_KEYS])

def svr_pca(nc=64, C=3): return make_pipeline(StandardScaler(), PCA(nc, random_state=SEED), SVR(C=C, epsilon=0.2, gamma="scale"))

class OrdinalExpect(BaseEstimator, RegressorMixin):
    """Multi-class logistic regression over the 0.5-grid labels; prediction = expected label value."""
    def __init__(self, nc=64, C=0.05, seed=42): self.nc, self.C, self.seed = nc, C, seed
    def fit(self, X, y_):
        self.pipe_ = make_pipeline(StandardScaler(), PCA(self.nc, random_state=self.seed), LogisticRegression(C=self.C, max_iter=3000))
        self.pipe_.fit(X, np.round(y_ * 2).astype(int)); return self
    def predict(self, X): return self.pipe_.predict_proba(X) @ (self.pipe_.classes_ / 2.0)

def make_models(seed=SEED):
    pca = lambda n=64: PCA(n, random_state=seed)
    M = {
        "ridge_w2v_l8":  (make_pipeline(StandardScaler(), pca(), RidgeCV(alphas=np.logspace(-1, 4, 20))), "emb"),
        "ridge_txt":     (ridge_model(), "txt"),
        "svr_txt":       (make_pipeline(StandardScaler(), SVR(C=3, epsilon=0.2, gamma="scale")), "txt"),
        "ridge_concat":  (ridge_model(), "concat"),
        "svr_concat":    (svr_pca(), "concat"),
        "svr_hand":      (make_pipeline(StandardScaler(), SVR(C=3, epsilon=0.2, gamma="scale")), "hand"),
        "gbr_hand":      (GradientBoostingRegressor(n_estimators=300, max_depth=3, learning_rate=0.03, subsample=0.8, random_state=seed), "hand"),
        "ord_whisper":   (OrdinalExpect(seed=seed), "whisper"),
        "ord_concat":    (OrdinalExpect(seed=seed), "concat"),
    }
    for s in ENCODERS:
        M[f"ridge_{s}"] = (ridge_model(), s)
        M[f"svr_{s}"]   = (svr_pca(), s)
    if HAS_LGBM:
        M["lgbm_hand"] = (LGBMRegressor(n_estimators=400, learning_rate=0.02, num_leaves=8, min_child_samples=15,
                                        subsample=0.8, subsample_freq=1, colsample_bytree=0.7, reg_lambda=5,
                                        random_state=seed, verbose=-1), "hand")
    return M

print(len(make_models()), "base learners:", list(make_models()))

### 6.3 One full pipeline per seed: OOF base models -> non-negative stack -> optional zero detector -> refit -> test prediction
Every seed uses different CV splits and different random states; test predictions are averaged over seeds.

In [ ]:
from sklearn.isotonic import IsotonicRegression
Z_train = np.hstack([views_train["whisper"], H_train.values]); Z_test = np.hstack([views_test["whisper"], H_test.values])
is0 = (y == 0).astype(int)
def zero_clf(): return make_pipeline(StandardScaler(), PCA(50, random_state=SEED), LogisticRegression(C=0.05, max_iter=5000))

def iso_fit(a, b): return IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=5).fit(a, b)

def choose_calibration(pred, folds3):
    """Nested CV over: none / linear rescale / isotonic. Returns (name, calibrated OOF)."""
    lin, iso = np.zeros(len(y)), np.zeros(len(y))
    for tr, va in folds3:
        lin[va] = LinearRegression().fit(pred[tr, None], y[tr]).predict(pred[va, None])
        iso[va] = iso_fit(pred[tr], y[tr]).predict(pred[va])
    cands = {"none": pred, "linear": np.clip(lin, 0, 5), "iso": np.clip(iso, 0, 5)}
    best = min(cands, key=lambda k: rmse(y, cands[k]))
    if rmse(y, cands["none"]) - rmse(y, cands[best]) < 0.002: best = "none"
    return best, cands[best]

def apply_calibration(name, oof_pred, new_pred):
    if name == "none":   return new_pred
    if name == "linear": return np.clip(LinearRegression().fit(oof_pred[:, None], y).predict(new_pred[:, None]), 0, 5)
    return np.clip(iso_fit(oof_pred, y).predict(new_pred), 0, 5)

def run_seed(seed):
    folds, folds2, folds3 = make_folds(seed), make_folds(seed + 1000), make_folds(seed + 2000)
    names = list(make_models(seed))
    oof = {k: np.zeros(len(y)) for k in names}
    for tr, va in folds:
        for k, (m, view) in make_models(seed).items():
            m.fit(views_train[view][tr], y[tr]); oof[k][va] = np.clip(m.predict(views_train[view][va]), 0, 5)
    oof_df = pd.DataFrame(oof); M = oof_df.values

    # level 1: non-negative linear stack with intercept
    stack = LinearRegression(positive=True).fit(M, y)
    stack_oof = np.zeros(len(y))
    for tr, va in folds2: stack_oof[va] = LinearRegression(positive=True).fit(M[tr], y[tr]).predict(M[va])
    stack_oof = np.clip(stack_oof, 0, 5)

    # optional zero detector, used only if it lowers OOF RMSE
    p0 = np.zeros(len(y))
    for tr, va in folds: p0[va] = zero_clf().fit(Z_train[tr], is0[tr]).predict_proba(Z_train[va])[:, 1]
    auc = roc_auc_score(is0, p0)
    S2 = np.column_stack([stack_oof, p0])
    lvl2_oof = np.zeros(len(y))
    for tr, va in folds2: lvl2_oof[va] = LinearRegression().fit(S2[tr], y[tr]).predict(S2[va])
    lvl2_oof = np.clip(lvl2_oof, 0, 5)
    use_zero = rmse(y, lvl2_oof) < rmse(y, stack_oof) - 0.002
    base_oof = lvl2_oof if use_zero else stack_oof

    # post-hoc calibration, chosen by nested CV
    calib, oof_final = choose_calibration(base_oof, folds3)

    # refit everything on all training data
    tr_parts, te_parts = [], []
    for k, (m, view) in make_models(seed).items():
        m.fit(views_train[view], y)
        tr_parts.append(np.clip(m.predict(views_train[view]), 0, 5)); te_parts.append(np.clip(m.predict(views_test[view]), 0, 5))
    tr_stack = np.clip(stack.predict(np.column_stack(tr_parts)), 0, 5)
    te_stack = np.clip(stack.predict(np.column_stack(te_parts)), 0, 5)
    if use_zero:
        zc = zero_clf().fit(Z_train, is0); lvl2 = LinearRegression().fit(S2, y)
        tr_pred = np.clip(lvl2.predict(np.column_stack([tr_stack, zc.predict_proba(Z_train)[:, 1]])), 0, 5)
        te_pred = np.clip(lvl2.predict(np.column_stack([te_stack, zc.predict_proba(Z_test)[:, 1]])), 0, 5)
    else:
        tr_pred, te_pred = tr_stack, te_stack
    tr_pred = apply_calibration(calib, base_oof, tr_pred); te_pred = apply_calibration(calib, base_oof, te_pred)

    print(f"seed {seed}: stack OOF={rmse(y, stack_oof):.4f} | zero AUC={auc:.3f} use_zero={use_zero} | calib={calib} | final OOF={rmse(y, oof_final):.4f}")
    return dict(oof_df=oof_df, oof=oof_final, train=tr_pred, test=te_pred, w=dict(zip(names, stack.coef_)), calib=calib)

runs = [run_seed(s) for s in tqdm(SEEDS, desc="seeds")]

## 7. Results: training RMSE (required) and diagnostics

In [ ]:
oof_final  = np.mean([r["oof"]   for r in runs], 0)
train_pred = np.mean([r["train"] for r in runs], 0)
test_pred  = np.mean([r["test"]  for r in runs], 0)
per_seed = [rmse(y, r["oof"]) for r in runs]

board = pd.DataFrame({"OOF RMSE": {k: np.mean([rmse(y, r["oof_df"][k]) for r in runs]) for k in runs[0]["oof_df"]},
                      "Pearson":  {k: np.mean([pearsonr(y, r["oof_df"][k])[0] for r in runs]) for k in runs[0]["oof_df"]}}).sort_values("OOF RMSE")
print(board.round(4))
w = pd.DataFrame([r["w"] for r in runs]).mean().sort_values(ascending=False); print("\nmean stack weights:", w[w > 1e-3].round(3).to_dict())

print("calibration chosen per seed:", [r["calib"] for r in runs])
print("=" * 72)
print(f"TRAINING RMSE (in-sample) = {rmse(y, train_pred):.4f}")
print(f"OOF RMSE per seed         = {np.round(per_seed, 4)}  (mean {np.mean(per_seed):.4f})")
print(f"OOF RMSE of seed-average  = {rmse(y, oof_final):.4f}   Pearson = {pearsonr(y, oof_final)[0]:.4f}   [v1 0.5930]")
print("=" * 72)
pd.DataFrame({"split": ["Training (in-sample)", "5-fold CV, mean over seeds"], "RMSE": [rmse(y, train_pred), np.mean(per_seed)],
              "Pearson": [pearsonr(y, train_pred)[0], np.mean([pearsonr(y, r["oof"])[0] for r in runs])]})

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
ax[0].scatter(y, oof_final, alpha=.5, s=15); ax[0].plot([0, 5], [0, 5], "r--")
ax[0].set(xlabel="True score", ylabel="Predicted (OOF)", title="OOF predictions vs truth")
res = oof_final - y
sns.histplot(res, kde=True, ax=ax[1]); ax[1].set(title=f"OOF residuals (mean {res.mean():.2f}, sd {res.std():.2f})")
pd.DataFrame({"true": y, "err": np.abs(res)}).groupby("true").err.mean().plot.bar(ax=ax[2], color="slateblue")
ax[2].set(ylabel="Mean |error|", title="Error by true score (check the 0.0 bar!)")
plt.tight_layout(); plt.show()

sq = pd.DataFrame({"true": y, "sq": res ** 2}).groupby(y == 0).sq.sum()
print(f"share of total squared error from the 0-label clips: {sq.get(True, 0) / sq.sum():.1%}")

cmp = pd.concat([board["OOF RMSE"], pd.Series({"FINAL (seed-avg)": rmse(y, oof_final)})]).sort_values()
cmp.plot.barh(color=["seagreen" if k.startswith("FINAL") else "teal" for k in cmp.index], figsize=(8, 7), title="OOF RMSE by model (lower is better)")
plt.xlabel("RMSE"); plt.tight_layout(); plt.show()

gb = GradientBoostingRegressor(n_estimators=300, max_depth=3, learning_rate=0.03, subsample=0.8, random_state=SEED).fit(H_train.values, y)
pd.Series(gb.feature_importances_, index=H_train.columns).sort_values()[-15:].plot.barh(figsize=(7, 5), color="darkorange", title="Gradient boosting: top-15 feature importances")
plt.tight_layout(); plt.show()

## 8. Predict test set & write submission

In [ ]:
sub = pd.DataFrame({"filename": test.filename, "label": test_pred})
assert len(sub) == len(test) and sub.label.between(0, 5).all() and sub.label.notna().all()
sub.to_csv("/content/submission.csv", index=False)
print(sub.shape, "| test pred:", sub.label.describe().round(3).to_dict())
print("train label:", train.label.describe().round(3).to_dict())
try:
    from google.colab import files; files.download("/content/submission.csv")
except Exception: pass
sub.head()

## 9. Discussion

* **Trust the OOF numbers**; the public leaderboard has only 216 clips and is noisy (v3: public 0.397 vs CV 0.497). Layer selection and stack weights share folds, so OOF is very slightly optimistic.
* **Encoders drive the score.** Check the model table: if a new encoder lands near the top and gets stack weight, keep it; if not, delete its line in `EXTRA_ENCODERS` and re-run (the cache makes that cheap).
* **Zero-label clips** are no longer a problem (near-zero share of squared error). Remaining error sits in low true scores (<= 2.0), which are rare and over-predicted.
* **Next ideas:** `whisper-large-v3` encoder, attention-pooling head on frame-level features, LoRA fine-tuning, per-model Optuna tuning.
* **Limitations:** Whisper repairs errors (CTC features partly compensate); CoLA is trained on written text; only 769 training clips.